In [ ]:
import os
import re
import sys
import json
import torch
import pickle
import contextlib
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict

from qiskit_aer import AerSimulator
from pytket.extensions.qiskit.backends.aer import AerBackend
# from qiskit.providers.aer import AerSimulator
# from pytket.extensions.qiskit import AerBackend


from lambeq.backend.grammar import Diagram, Id
from lambeq import (
    AtomicType,
    IQPAnsatz,
    RemoveCupsRewriter,
    SimpleRewriteRule,
    Rewriter,
    UnifyCodomainRewriter,
    DepCCGParser
)

# import depccg
# from lambeq import ( CCGParser, CCGTree, CCGRuleUseError, CCGRule, CCGType,
#                     CCGBankParseError, CCGBankParser, DepCCGParseError )


In [ ]:
import logging
logging.getLogger("allennlp").setLevel(logging.WARNING)
logging.getLogger("depccg").setLevel(logging.WARNING)
parser = DepCCGParser(model='elmo', device=0) # device=:  -1 == CPU | 0 == GPU | 1 == second GPU

In [ ]:
def get_deep_type(obj):
    if isinstance(obj, list):
        # We look at the unique types inside the list to keep it readable
        inner_types = {get_deep_type(item) for item in obj}
        return f"List[{' | '.join(sorted(inner_types))}]"

    elif isinstance(obj, dict):
        # We summarize the types of all keys and all values
        key_types = {get_deep_type(k) for k in obj.keys()}
        val_types = {get_deep_type(v) for v in obj.values()}
        return f"Dict[{' | '.join(sorted(key_types))}, {' | '.join(sorted(val_types))}]"

    else:
        # Return the class name (e.g., 'Diagram' or 'str')
        return type(obj).__name__
    
from collections.abc import Mapping, Sequence

def get_deep_shape(obj, level=0):
    indent = "  " * level

    # 1. Atomic types (Strings/Bytes) - Removed len() for brevity
    if isinstance(obj, (str, bytes)):
        return f"{indent}str"

    # 2. Sequences (Lists, Tuples, etc.)
    elif isinstance(obj, Sequence):
        if not obj:
            return f"{indent}{type(obj).__name__}(len=0)"

        header = f"{indent}{type(obj).__name__}(len={len(obj)})"
        
        # Calculate shapes of all children to check for uniformity
        child_shapes = [get_deep_shape(item, level + 1).lstrip() for item in obj]
        unique_shapes = sorted(list(set(child_shapes)))

        if len(unique_shapes) == 1:
            # All items are identical structure (e.g., all are 'str')
            return f"{header}\n{indent}  [*] -> {unique_shapes[0]}"
        else:
            # Items differ; list them individually
            # Note: For massive lists with mixed types, you might want to 
            # only show the first few, but here we show all as requested.
            lines = [header]
            for i, shape in enumerate(child_shapes):
                lines.append(f"{indent}  [{i}] -> {shape}")
            return "\n".join(lines)

    # 3. Dictionaries
    elif isinstance(obj, Mapping):
        if not obj:
            return f"{indent}dict(len=0)"

        lines = [f"{indent}dict(len={len(obj)})"]
        for key, value in obj.items():
            child = get_deep_shape(value, level + 1).lstrip()
            lines.append(f"{indent}  key={repr(key)} -> {child}")
        return "\n".join(lines)

    # 4. Base objects
    else:
        return f"{indent}{type(obj).__name__}"

def find_mismatches(data_a, data_b):
    # 1. Check if the outer lists are even the same length
    if len(data_a) != len(data_b):
        print(
            f"❌ [CRITICAL] Outer List Length Mismatch: List A={len(data_a)}, List B={len(data_b)}"
        )

    # Iterate through the top-level list
    for i, (dict_a, dict_b) in enumerate(zip(data_a, data_b)):
        # Check if the keys in the dictionaries match
        keys_a = set(dict_a.keys())
        keys_b = set(dict_b.keys())

        if keys_a != keys_b:
            print(f"❌ [Index {i}] Key Mismatch:")
            print(f"   Keys only in A: {keys_a - keys_b}")
            print(f"   Keys only in B: {keys_b - keys_a}")
            continue  # Skip to next list item if keys don't match

        # Check values for each key
        for key in keys_a:
            list_a = dict_a[key]
            list_b = dict_b[key]

            # 2. Check lengths of the lists inside the dictionary
            if len(list_a) != len(list_b):
                print(f"❌ [Index {i}][Key: '{key}'] Inner List Length Mismatch:")
                print(f"   Length A: {len(list_a)}")
                print(f"   Length B: {len(list_b)}")

            # 3. Check individual elements inside those lists
            # This handles List[Diagram], List[List[int]], and List[str]
            for j, (val_a, val_b) in enumerate(zip(list_a, list_b)):
                if val_a != val_b:
                    print(
                        f"❌ [Index {i}][Key: '{key}'][Inner Index {j}] Content Mismatch!"
                    )
                    print(f"   Type A: {type(val_a).__name__}")
                    print(f"   Type B: {type(val_b).__name__}")

                    # If they are small (like List[int] or str), print the actual value
                    if not hasattr(
                        val_a, "draw"
                    ):  # Don't print full Diagrams, they are too big
                        print(f"   Value A: {val_a}")
                        print(f"   Value B: {val_b}")
                    else:
                        print(
                            f"   (Diagram content differs - possibly different boxes or wires)"
                        )

def diagnose_variable(variable):
    # print("TOP LEVEL LENGTH:", len(variable) if hasattr(variable, '__len__') else "N/A")
    print("DEEP TYPE:")
    print(get_deep_type(variable))
    print("\nDEEP SHAPE:")
    print(get_deep_shape(variable))


In [ ]:
import statistics

def print_length(dataset):
    dict_string = "text_sentences"
    if "circuits" in list(dataset[0].keys()):
        dict_string = "circuits"

    count = 0
    count_labels = 0

    for data in dataset:
        count += len(data[dict_string])
        count_labels += len(data["labels"])
    
    print(f"n_{dict_string}: {count} | n_lables: {count_labels}")

def print_errors(errors):
    print(f"error_list_length: {len(errors)}")
    for i, error in enumerate(errors):
        print(f"{i}: {error}")
    print

def print_n_qubits(n_qubits_list):
    print(f"qubit_list: {n_qubits_list}")
    print(f"qubit_list_length: {len(n_qubits_list)}")
    print(f"qubit_list_mean: {statistics.mean(n_qubits_list)}\n")
    
    

In [ ]:
def combine_articles(dataset) -> List[Dict]:
    combined_text_sentences, combined_labels = [], []

    for article in dataset:
        # zip(combined, article["text_sentences"]) ####### paklaust gpt kam naudojamas zip ir kodel jis cia neveikia, i.e. kaip jis cia tiksliai veikia?
        # zip(labels, article["labels"])

        combined_text_sentences += article["text_sentences"]
        combined_labels += article["labels"]

    combined_dataset = [{
        "text_sentences": combined_text_sentences,
        "labels": combined_labels,
    }]

    return combined_dataset

def combine_n_articles(dataset, n_to_merge: int = 3) -> List[Dict]:
    combined_dataset = []
    
    for i in range(0, len(dataset), n_to_merge):
        chunk = dataset[i : i + n_to_merge]
        combined_dataset.extend(combine_articles(chunk))
        
    return combined_dataset


def load_PreSumm_pts(left=0, right=144, ds_purpose = "train", calculate_articles: bool = False) -> List[Dict]:
    raw_ds = []
    k = left
    n_articles = []
    
    while k < right:
        print(f"load_PreSumm_pts_{k}")
        file_path = f"Dataset/Raw/cnn_dailymail/_PreSumm/cnndm.{ds_purpose}.{k}.bert.pt"
        loaded_data = torch.load(file_path)

        for i, file_element in enumerate(loaded_data):

            quantum_state_distribution_labels = []
            for label in file_element["src_sent_labels"]:
                if label:
                    quantum_state_distribution_labels.append([0,1])
                else:
                    quantum_state_distribution_labels.append([1,0])

            raw_ds.append(
                {
                    "text_sentences": file_element["src_txt"],
                    # "org_labels": line["src_sent_labels"],
                    "labels": quantum_state_distribution_labels
                }
            )
        n_articles.append(len(loaded_data))
        k += 1
        
    return (raw_ds, n_articles) if calculate_articles else raw_ds

HIGHEST_FILENAME_NUMBER = 8

def read_PreSum_multiple(amount: int = 4, dataset_purpose: str = "train"):
    left = 0
    right = amount
    loaded_combined_dataset = []

    while left < HIGHEST_FILENAME_NUMBER:
        print(f"read_PreSumm_multiple_{left}")
        loaded_combined_dataset.append(
            combine_articles( load_PreSumm_pts(left, right, dataset_purpose) )
        )
        left = right
        right = min(HIGHEST_FILENAME_NUMBER, left + amount)

    return loaded_combined_dataset


In [ ]:
# https://chatgpt.com/s/t_69b2c7f7e28081919384bb842e7c46b3 - apie sakiniu preprocesinga kuris sutrumpina 40-60 proc sakiniu ilgio

In [ ]:
_CLEAN_REGEX = re.compile(r"[^\w\s'-]")

ansatz    = IQPAnsatz(
    {AtomicType.SENTENCE: 1,
     AtomicType.NOUN:     1,
     AtomicType.PREPOSITIONAL_PHRASE: 0,
     AtomicType.ADJECTIVE: 0,
     AtomicType.ADVERB: 0},
    n_layers=2, n_single_qubit_params=3
)

def create_rewriter():
    # Rule to delete conjunction boxes (“and”, “but”) # just the wire, no box
    conj_rule = SimpleRewriteRule(cod=AtomicType.CONJUNCTION, template=Id(AtomicType.CONJUNCTION))

    # Rule to delete punctuation boxes (commas, quotes, dashes)
    # punc_rule = SimpleRewriteRule(cod=AtomicType.PUNCTUATION, template=Id(AtomicType.PUNCTUATION))

    # remove_pp2 = SimpleRewriteRule(cod=AtomicType.PREPOSITIONAL_PHRASE, template=Id(AtomicType.SENTENCE))
    # remove_pp = SimpleRewriteRule(cod=AtomicType.PREPOSITIONAL_PHRASE, template=Id(AtomicType.NOUN))

    # rewriter = Rewriter(
    #     [
    #         'coordination', 'determiner',
    #         'postadverb', 'preadverb',
    #         'connector', 'auxiliary',
    #         'prepositional_phrase',
    #         'subject_rel_pronoun',
    #         'object_rel_pronoun',
    #         'adjective', 'noun_phrase',
    #     ]
    # )

    Rewriter([
        'determiner',
        'auxiliary',
        'connector',
        'coordination',
        'subject_rel_pronoun',
        'object_rel_pronoun'
    ])
    
    rewriter.add_rules(conj_rule)
    return rewriter

rewriter = create_rewriter()
remove_cups = RemoveCupsRewriter()
unify = UnifyCodomainRewriter(output_type=AtomicType.SENTENCE)

In [ ]:
simulator = AerSimulator(
    method="statevector", device="GPU",
    precision="single",         # 32-bit float for ~2× speedup on large statevectors
    cuStateVec_enable=True  ,     # turn on NVIDIA cuStateVec kernels
    # batched_shots_gpu=True,     # batch thousands of shots very efficiently on GPU
    # batched_shots_gpu_max_qubits=29, # 8 ⋅ 2^n == 7 ⋅ 2^30, n ~= 29.8
    # num_threads_per_device=2    # limit CPU threads per GPU to reduce overhead
)

backend = AerBackend(simulation_method="statevector")
backend._qiskit_backend = simulator

# comp_pass = backend.default_compilation_pass(2)

In [ ]:
#############################################
# Step 2. Preprocessing and lambeq Pipeline
#############################################


def remove_by_idx(ls, remove):
    if not remove:
        return ls
    remove.sort(reverse=True)
    for n in remove:
        ls.pop(n)
    return ls


def sentence_simplify(sentences):
    return [_CLEAN_REGEX.sub("", s) for s in sentences if s is not None]

def sent2diagrams_single(sentences):
    diagrams, none_idx = [], []
    for i, sent in enumerate(sentences):
        d = parser.sentence2diagram(sent, tokenised=False, suppress_exceptions=True)
        if d is None:
            none_idx.append(i)
            continue

        diagrams.append(d)
    print(f"Whilst parsing sentences2diagrams, lost {len(sentences) - len(diagrams)} due to Null, out of {len(sentences)}.")

    return diagrams, none_idx

def sent2diagrams(sentences):
    none_idx = []
    
    diagrams = parser.sentences2diagrams(
        sentences, tokenised=False, suppress_exceptions=True
    )
    for i, diag in enumerate(diagrams):
        if diag is None:
            none_idx.append(i)

    print(
        f"Whilst parsing sentences2diagrams, lost {len(none_idx)} due to Null, out of {len(sentences)}."
    )

    return diagrams, none_idx


def normalize(sentence_diagrams):
    diagrams_normalized, none_idx, errs = [], [], []
    drop_rewrite = 0
    drop_cups = 0
    for i, d in enumerate(sentence_diagrams):
        try:
            d = rewriter(d)
            if d is None:
                none_idx.append(i)
                drop_rewrite += 1
                continue

            d = remove_cups(d)
            d = d.normal_form()
            d = d.pregroup_normal_form()
            d = unify(d)

        except Exception as e:
            none_idx.append(i)
            errs.append(f"normalize() | {e}")
            drop_cups += 1
            continue

        diagrams_normalized.append(d)

    print(
        f"Dropped {drop_rewrite} diagrams in rewrite, {drop_cups} in cup removal, out of {len(sentence_diagrams)}."
    )

    return diagrams_normalized, none_idx, errs


def quantum_encode(diagrams: "List"):
    encoded_diagrams, remove, errs = [], [], []
    for i, diagram in enumerate(diagrams):
        try:
            circ = ansatz(diagram)
            # circ = circ.to_tk()               ########### gal reiktu i tk circuit convertint??? kas is vis yra tas tk circuit????
            encoded_diagrams.append(circ)
        except Exception as e:
            errs.append(f"quantum_encode() | {e}")
            remove.append(i)

    return encoded_diagrams, remove, errs


def will_train(
    circuits,
    qubit_limit: int = 28,
    mem_limit_bytes: int = 7 * 2**30,  # 7 GiB
):

    valid, invalid_idxs, errs = [], [], []
    n_qubits_list = []

    for idx, circ in enumerate(circuits):
        try:
            tk_circ = circ.to_tk()
            n_qubits = tk_circ.n_qubits
            n_qubits_list.append(n_qubits)

            if n_qubits > qubit_limit:
                raise RuntimeError(
                    f"{n_qubits} qubits exceeds safe limit {qubit_limit}."
                )

            needed = 8 * (2**n_qubits)
            if needed > mem_limit_bytes:
                raise RuntimeError(
                    f"Needs {needed} bytes > limit {mem_limit_bytes} bytes ({needed / 2**20:.0f} MiB > {(mem_limit_bytes / 2**20):.0f} MiB). "
                )
                # raise RuntimeError(f"Needs {needed/2**30:.2f} GiB > limit {mem_limit_bytes/2**30:.2f} GiB.")

            # syms = tk_circ.free_symbols()
            # if syms:
            #     bind_map = {s: 0.0 for s in syms}
            #     tk_circ.symbol_substitution(bind_map)

            compiled = backend.get_compiled_circuit(tk_circ)
 
            # unnecessary and expensive due to parameterization which does not need shots.
            # simulator.run(tk_circ, n_shots=1) 
            # backend.run_circuit(tk_circ, n_shots=1)

        except Exception as e:
            invalid_idxs.append(idx)
            errs.append(f"will_train() | {e}")
            continue

        valid.append(circ)

    return valid, invalid_idxs, errs, n_qubits_list


def preprocess_and_encode(dataset):
    encoded_data, errors = [], []
    n_sentences = 0
    n_circuits = 0

    # for i, data_dict in enumerate(tqdm(dataset, desc="Filtering and Encoding dataset")):
    #     with open(os.devnull, 'w') as devnull, \
    #      contextlib.redirect_stdout(devnull), \
    #      contextlib.redirect_stderr(devnull):

    for i, data_dict in enumerate(dataset):
            text_sentences = data_dict["text_sentences"]
            labels         = data_dict["labels"]

            n_sentences += len(text_sentences)

            sentences_simplified = sentence_simplify(text_sentences)
            diagrams, remove     = sent2diagrams(sentences_simplified)
            diagrams             = remove_by_idx(diagrams, remove)
            text_sentences       = remove_by_idx(text_sentences, remove)
            labels               = remove_by_idx(labels, remove)

            normalized_diagrams, remove, errs2 = normalize(diagrams)
            text_sentences                     = remove_by_idx(text_sentences, remove)
            labels                             = remove_by_idx(labels, remove)

            circuits, remove, errs3 = quantum_encode(normalized_diagrams)
            text_sentences          = remove_by_idx(text_sentences, remove)
            labels                  = remove_by_idx(labels, remove)

            circuits, remove, errs4, n_qubits = will_train(circuits)
            text_sentences          = remove_by_idx(text_sentences, remove)
            labels                  = remove_by_idx(labels, remove)

            encoded_data.append(
                {
                    "circuits": circuits,
                    "labels": labels,
                    "original_text_sentences": text_sentences,
                }
            )

            errors += errs2 + errs3 + errs4 # + errs1 sent2diagrams has suppress_exceptions=True, so instead of errors, it returns None
            n_circuits += len(circuits)

    errors.append(f"n_sentences: {n_sentences} | n_circuits: {n_circuits} | {n_sentences - n_circuits}")

    return encoded_data, errors, n_qubits



In [ ]:
ld_ds, n_articles = load_PreSumm_pts(right=1, calculate_articles=True)
print(sum(n_articles))

In [ ]:
ld_ds_combined = combine_n_articles(ld_ds[:30], 3)
ld_ds_encoded, ld_ds_errors, ld_ds_n_qubits = preprocess_and_encode(ld_ds_combined)


In [ ]:
encoded = {
    "encoded_dataset": ld_ds_encoded,
    "errors": ld_ds_errors,
    "n_qubits": ld_ds_n_qubits
}
with open('Dataset/Encoded/cnn_dailymail/PreSumm_0_30_3.pkl', 'wb') as file:
    pickle.dump(encoded, file)

In [ ]:
diagnose_variable(ld_ds_combined)

In [ ]:
diagnose_variable(ld_ds_encoded)

In [ ]:
print_length(ld_ds_combined)
print_length(ld_ds_encoded)

print_errors(ld_ds_errors)

In [ ]:
ld_ds_combined3 = combine_n_articles(ld_ds[:30], 10)
ld_ds_encoded3, ld_ds_errors3, ld_ds_n_qubits3 = preprocess_and_encode(ld_ds_combined3)

In [ ]:
encoded3 = {
    "encoded_dataset": ld_ds_encoded3,
    "errors": ld_ds_errors3,
    "n_qubits": ld_ds_n_qubits3
}
with open('Dataset/Encoded/cnn_dailymail/PreSumm_0_30_10.pkl', 'wb') as file:
    pickle.dump(encoded3, file)

In [ ]:
print_length(ld_ds_combined3)
print_length(ld_ds_encoded3)

print_errors(ld_ds_errors3)

In [ ]:
def s2d(raw_ds):
    diagrams = []
    for data_dict in tqdm(raw_ds):
        with open(os.devnull, 'w') as devnull, \
         contextlib.redirect_stdout(devnull), \
         contextlib.redirect_stderr(devnull):
            # diagrams.append(sent2diagrams(data_dict["text_sentences"]))
            sent2diagrams(data_dict["text_sentences"])

    return diagrams

x_ds = combine_n_articles(ld_ds[:200], 1)
d = s2d(x_ds)

# 32m 50s single articles over 200 in sent2diagrams
# 37m 28s 5:1 article over 200 in sent2diagrams
# 43m 24s 20:1 articles over 200 in sent2diagrams
# 84m 19s single articles over 200 in sent2diagrams_single

In [ ]:
get_deep_type(x_ds)

In [ ]:
encoded_dataset, errors, n_qubits = preprocess_and_encode(ld_ds_combined)
# On GPU it took ~4m 36.0s to encode first 10 PreSumm articles seperately without will_train()
# On GPU it took ~14m 39.4s to encode first 30 PreSumm articles seperately without will_train()
# On GPU it took ~14m 27.1s to encode first 30 PreSumm articles (merged 2:1) without will_train()
# On GPU it took ~14m 27.6s to encode first 30 PreSumm articles (merged 3:1) without will_train()
# On GPU it took ~14m 54.1s to encode first 30 PreSumm articles (merged 6:1) without will_train()
# On GPU it took ~15m 40.0s to encode first 30 PreSumm articles (merged 10:1) without will_train()

# On GPU it took ~24m 30.0s to encode first 30 PreSumm articles seperately
# On GPU it took ~24m 32.6s to encode first 30 PreSumm articles (merged 2:1)
# On GPU it took ~24m 24.7s to encode first 30 PreSumm articles (merged 3:1)
# On GPU it took ~24m 46.4s to encode first 30 PreSumm articles (merged 6:1)
# On GPU it took ~25m 50.s to encode first 30 PreSumm articles (merged 10:1)
# On GPU it took ~27m 54.s to encode first 30 PreSumm articles (merged 30:1)

# On GPU it took ~24m 14s to encode first 30 PreSumm articles seperately with optimized will_train and AerSimulator object
# On GPU it took ~24m 30s to encode first 30 PreSumm articles (merged 2:1) with optimized will_train and AerSimulator object
# On GPU it took ~23m 56s to encode first 30 PreSumm articles (merged 3:1) with optimized will_train and AerSimulator object 2
# On GPU it took ~15m 46s to encode first 30 PreSumm articles (merged 3:1) with optimized will_train and AerSimulator object 1
# On GPU it took ~24m 54s to encode first 30 PreSumm articles (merged 6:1)  with optimized will_train and AerSimulator object 2
# On GPU it took ~17m 1s to encode first 30 PreSumm articles (merged 6:1)  with optimized will_train and AerSimulator object 1
# On GPU it took ~26m 0s to encode first 30 PreSumm articles (merged 10:1)  with optimized will_train and AerSimulator object 2
# On GPU it took ~m .s to encode first 30 PreSumm articles (merged 30:1)  with optimized will_train and AerSimulator object

In [ ]:
print_length(encoded_dataset)
print_n_qubits(n_qubits)
print_errors(errors)

In [ ]:
encoded = {
    "encoded_dataset": encoded_dataset,
    "errors": errors,
    "n_qubits": n_qubits
}
with open('Dataset/Encoded/cnn_dailymail/PreSumm_0_30_1.pkl', 'wb') as file:
    pickle.dump(encoded, file)

In [ ]:
with open('Dataset/Encoded/cnn_dailymail/PreSumm_0_30_1.pkl', "rb") as file:
    encoded_loaded = pickle.load(file)

In [ ]:
diagnose_variable(encoded_dataset)

In [ ]:
import gc

# Delete large temporary variables
# del expensive_tensors

# Force Python to find unreferenced objects
gc.collect()

# Force the GPU to release the cached memory pool
torch.cuda.empty_cache()

In [ ]:
print_errors(errors)
print_n_qubits(n_qubits)

In [ ]:
# 87
# 0: Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ p @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ s @ s.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ p @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ s @ s.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 1: Diagram 0 (cod=n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n.l.l @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n.l.l @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 2: Diagram 0 (cod=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 3: Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 4: 37 qubits exceeds safe limit 28. ( will_train() )
# 86: 40 qubits exceeds safe limit 28. ( will_train() )
# [37, 31, 53, 13, 23, 14, 14, 14, 13, 17, 13, 7, 18, 13, 20, 17, 5, 21, 13, 9, 13, 11, 9, 17, 10, 27, 15, 14, 7, 18, 22, 16, 20, 13, 9, 34, 12, 13, 23, 26, 45, 14, 21, 7, 19, 7, 52, 12, 9, 17, 18, 13, 12, 26, 23, 16, 12, 11, 12, 11, 27, 26, 32, 38, 31, 26, 43, 29, 5, 5, 18, 24, 36, 20, 20, 35, 24, 40, 18, 22, 23, 42, 25, 12, 23, 28, 15, 17, 38, 49, 36, 22, 13, 27, 29, 18, 29, 16, 15, 31, 26, 20, 6, 17, 19, 32, 5, 36, 22, 18, 17, 8, 18, 16, 26, 11, 13, 14, 15, 7, 8, 5, 11, 18, 24, 17, 14, 5, 12, 11, 7, 29, 11, 34, 20, 38, 24, 13, 26, 25, 22, 38, 49, 24, 46, 21, 36, 16, 8, 13, 35, 17, 41, 33, 33, 31, 35, 49, 22, 24, 51, 37, 41, 36, 37, 32, 28, 22, 27, 28, 16, 43, 20, 17, 30, 27, 47, 49, 22, 36, 31, 40, 25, 17, 27, 47, 53, 46, 25, 15, 13, 18, 24, 22, 34, 31, 21, 18, 20, 43, 59, 13, 29, 12, 14, 20, 15, 47, 46, 31, 15, 38, 50, 17, 39, 23, 28, 42, 16, 49, 42, 32, 49, 21, 26, 51, 18, 11, 41, 23, 33, 30, 5, 11, 16, 14, 21, 27, 22, 13, 27, 23, 30, 27, 28, 35, 34, 23, 16, 11, 22, 29, 5, 10, 18, 24, 20, 21, 42, 28, 50, 36, 42, 14, 13, 27, 42, 40, 16]
# length: 269
# 23.966542750929367

In [ ]:
# ld_ds[:30], 3
# 60
# 0: Diagram 0 (cod=n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n.l.l @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n.l.l @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 1: Diagram 0 (cod=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 2: Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 3: 38 qubits exceeds safe limit 28. ( will_train() )
# 59: 40 qubits exceeds safe limit 28. ( will_train() )
# [22, 38, 49, 24, 46, 21, 36, 16, 8, 13, 35, 17, 41, 33, 33, 31, 35, 49, 22, 24, 51, 37, 41, 36, 37, 32, 28, 22, 27, 28, 16, 43, 20, 17, 30, 27, 47, 49, 22, 36, 31, 40, 25, 17, 27, 47, 53, 46, 25, 15, 13, 18, 24, 22, 34, 31, 21, 18, 20, 43, 59, 13, 29, 12, 14, 20, 15, 47, 46, 31, 15, 38, 50, 17, 39, 23, 28, 42, 16, 49, 42, 32, 49, 21, 26, 51, 18, 11, 41, 23, 33, 30, 5, 11, 16, 14, 21, 27, 22, 13, 27, 23, 30, 27, 28, 35, 34, 23, 16, 11, 22, 29, 5, 10, 18, 24, 20, 21, 42, 28, 50, 36, 42, 14, 13, 27, 42, 40, 16]
# length: 129
# 28.45736434108527

In [ ]:
# 149
# 0: Diagram 0 (cod=n @ n.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ n.l @ n) does not compose with diagram 1 (dom=n @ n.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ n.l @ n @ n) ( normalize() )
# 1: Diagram 0 (cod=n @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ p.l) does not compose with diagram 1 (dom=n @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ p.l @ n) ( normalize() )
# 2: Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ p @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ s @ s.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ p @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ s @ s.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 3: Diagram 0 (cod=n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n.l.l @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n.l.l @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 4: Diagram 0 (cod=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 5: Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 6: 29 qubits exceeds safe limit 28. ( will_train() )
# 148: 40 qubits exceeds safe limit 28. ( will_train() )
# [5, 5, 26, 29, 30, 19, 58, 30, 35, 34, 55, 20, 71, 26, 30, 59, 26, 16, 25, 17, 40, 39, 30, 28, 32, 25, 5, 5, 18, 30, 32, 5, 11, 11, 45, 39, 26, 29, 26, 13, 23, 7, 18, 32, 14, 11, 26, 23, 15, 6, 4, 11, 14, 18, 7, 14, 20, 14, 35, 35, 14, 26, 25, 16, 13, 25, 12, 14, 18, 24, 35, 36, 19, 15, 10, 47, 31, 17, 16, 33, 24, 18, 30, 32, 37, 41, 36, 33, 18, 52, 45, 57, 46, 28, 38, 14, 20, 13, 47, 27, 42, 38, 31, 36, 24, 53, 15, 34, 40, 12, 31, 30, 5, 5, 43, 45, 48, 27, 10, 18, 50, 38, 30, 76, 19, 19, 20, 22, 29, 28, 23, 23, 30, 19, 28, 16, 30, 37, 25, 39, 37, 31, 53, 13, 23, 14, 14, 14, 13, 17, 13, 7, 18, 13, 20, 17, 5, 21, 13, 9, 13, 11, 9, 17, 10, 27, 15, 14, 7, 18, 22, 16, 20, 13, 9, 34, 12, 13, 23, 26, 45, 14, 21, 7, 19, 7, 52, 12, 9, 17, 18, 13, 12, 26, 23, 16, 12, 11, 12, 11, 27, 26, 32, 38, 31, 26, 43, 29, 5, 5, 18, 24, 36, 20, 20, 35, 24, 40, 18, 22, 23, 42, 25, 12, 23, 28, 15, 17, 38, 49, 36, 22, 13, 27, 29, 18, 29, 16, 15, 31, 26, 20, 6, 17, 19, 32, 5, 36, 22, 18, 17, 8, 18, 16, 26, 11, 13, 14, 15, 7, 8, 5, 11, 18, 24, 17, 14, 5, 12, 11, 7, 29, 11, 34, 20, 38, 24, 13, 26, 25, 22, 38, 49, 24, 46, 21, 36, 16, 8, 13, 35, 17, 41, 33, 33, 31, 35, 49, 22, 24, 51, 37, 41, 36, 37, 32, 28, 22, 27, 28, 16, 43, 20, 17, 30, 27, 47, 49, 22, 36, 31, 40, 25, 17, 27, 47, 53, 46, 25, 15, 13, 18, 24, 22, 34, 31, 21, 18, 20, 43, 59, 13, 29, 12, 14, 20, 15, 47, 46, 31, 15, 38, 50, 17, 39, 23, 28, 42, 16, 49, 42, 32, 49, 21, 26, 51, 18, 11, 41, 23, 33, 30, 5, 11, 16, 14, 21, 27, 22, 13, 27, 23, 30, 27, 28, 35, 34, 23, 16, 11, 22, 29, 5, 10, 18, 24, 20, 21, 42, 28, 50, 36, 42, 14, 13, 27, 42, 40, 16]
# length: 409
# 24.9119804400978

In [ ]:
# 14
# 0: ( will_train() ) 51 qubits exceeds safe limit 28.
# 13: ( will_train() ) 40 qubits exceeds safe limit 28.
# [21, 26, 51, 18, 11, 41, 23, 33, 30, 5, 11, 16, 14, 21, 27, 22, 13, 27, 23, 30, 27, 28, 35, 34, 23, 16, 11, 22, 29, 5, 10, 18, 24, 20, 21, 42, 28, 50, 36, 42, 14, 13, 27, 42, 40, 16]
# length: 46
# 24.695652173913043

In [ ]:
# 38
# 0: ( normalize() ) Diagram 0 (cod=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj)
# 1: ( normalize() ) Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj)
# 2: ( will_train() ) 36 qubits exceeds safe limit 28.
# 37: ( will_train() ) 40 qubits exceeds safe limit 28.
# [36, 31, 40, 25, 17, 27, 47, 53, 46, 25, 15, 13, 18, 24, 22, 34, 31, 21, 18, 20, 43, 59, 13, 29, 12, 14, 20, 15, 47, 46, 31, 15, 38, 50, 17, 39, 23, 28, 42, 16, 49, 42, 32, 49, 21, 26, 51, 18, 11, 41, 23, 33, 30, 5, 11, 16, 14, 21, 27, 22, 13, 27, 23, 30, 27, 28, 35, 34, 23, 16, 11, 22, 29, 5, 10, 18, 24, 20, 21, 42, 28, 50, 36, 42, 14, 13, 27, 42, 40, 16]
# length: 90
# 27.42222222222222

In [ ]:
# on CPU it takes ~5m 56.5s to encode first 10 articles
# on `GPU` it takes ~4m 57.5s to encode first 10 articles

# on CPU and feeding whole articles (not single sentences) it takes ~3m 4.2s to encode first 10 articles
# on `GPU` and feeding whole articles (not single sentences) it takes ~2m 44.2s to encode first 10 articles

# on CPU and feeding multiple (40) articles it takes ~_m _._s
# on `GPU` and feeding multiple (40) articles it takes ~15m 21.3s

# on CPU and feeding whole articles (not single sentences) it takes ~m s to encode first 40 articles
# on `GPU` and feeding whole articles (not single sentences) it takes ~14m 49s to encode first 40 articles
